# Priors for `lambda` and `delta_lambda_aero` in cscm-calibrate

**Scope.** This notebook produces drop-in `[low, high]` flat-prior bounds for two CICERO-SCM parameters that the cscm-calibrate pipeline needs to sample over:

1. `lambda` — climate-sensitivity parameter (K per W m⁻²); the inverse of the equilibrium Gregory feedback.
2. `delta_lambda_aero` — pattern-mediated feedback sensitivity (W m⁻² K⁻¹); modulates the feedback as $\lambda_\text{eff}(t) = \lambda_0 + \Delta\lambda_\text{aero}\,w_\text{aero}(t)$, where $w_\text{aero}$ is the magnitude-weighted aerosol forcing fraction.

**What this notebook is not.** It is *not* a calibration. The cscm-calibrate pipeline already has GMST, OHC, ERFaer, atmospheric CO₂ and carbon-flux constraints and sweeps the full set of CICERO-SCM parameters (ocean coupling, mixed-layer depth, aerosol forcing efficacies, …). All this notebook does is decide where to *start* sampling each of these two parameters; the pipeline does the actual posterior shaping.

**Sources used:**
- Andrews et al. (2022) Table 4 — DEEP-C v5 diagnoses of $-\lambda_\text{hist}$ over various 30-year satellite-era windows (1985-2019).
- IPCC AR6 (Forster et al. 2021) — assessed long-term feedback parameter $-\lambda_\text{4xCO}_2 = 1.16 \pm 0.65$ W m⁻² K⁻¹ (5-95%).

**Sign convention warning.** Andrews uses $\lambda$ negative for stabilising feedback. The CSV column `neg_lambda` is the magnitude $-\lambda$ (positive). CICERO-SCM internally uses positive Gregory feedback (= $-\lambda$ in Andrews units). Numbers below are in the positive-stabilising convention throughout. Reminder: `pamset_udm['lambda']` is the *climate-sensitivity parameter* (K per W m⁻²); the Gregory feedback is $1/\lambda$.

## Setup

In [1]:
import os
import sys
import json

import numpy as np
import pandas as pd

REPO_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))
sys.path.insert(0, os.path.join(REPO_ROOT, 'src'))
TEST_DATA = os.path.join(REPO_ROOT, 'tests', 'test-data')

from ciceroscm import CICEROSCM
from ciceroscm.input_handler import (
    read_components, read_inputfile, read_natural_emissions,
)

PINATUBO_YEARS = (1991, 1992)

## Andrews et al. (2022) Table 4

Six 30-year satellite-era windows (DEEP-C v5) with diagnosed $-\lambda_\text{hist}$ ranging 1.44-1.98 W m⁻² K⁻¹. We focus on v5 (Andrews' principal data set); v3 vs v5 disagreement gives a structural-uncertainty floor.

In [2]:
andrews = pd.read_csv(os.path.join(REPO_ROOT, 'notebooks', 'andrews_constraints.csv'))
andrews.columns = ['version', 'y0', 'y1', 'psi', 'neg_lambda', 'kappa']
v5 = andrews[andrews['version'] == 'DEEP-C v5'].copy()
v5['window'] = v5['y0'].astype(str) + '-' + v5['y1'].astype(str)
print('DEEP-C v5 windows:')
print(v5[['window', 'neg_lambda']].to_string(index=False))
neg_lam_min, neg_lam_max = float(v5['neg_lambda'].min()), float(v5['neg_lambda'].max())
neg_lam_mean = float(v5['neg_lambda'].mean())
print(f'\nv5 -lambda_hist range: {neg_lam_min:.2f} to {neg_lam_max:.2f}, mean {neg_lam_mean:.2f} W m^-2 K^-1')
print(f'v3 vs v5 (1985-2014) structural offset: 2.24 - 1.98 = 0.26 W m^-2 K^-1')

DEEP-C v5 windows:
   window  neg_lambda
1985-2014        1.98
1986-2015        1.75
1987-2016        1.55
1988-2017        1.62
1989-2018        1.66
1990-2019        1.44

v5 -lambda_hist range: 1.44 to 1.98, mean 1.67 W m^-2 K^-1
v3 vs v5 (1985-2014) structural offset: 2.24 - 1.98 = 0.26 W m^-2 K^-1


## Prior on `lambda` (= 1/Gregory feedback)

IPCC AR6 (Forster et al. 2021) assesses the long-term feedback parameter at $-\lambda  = 1.16 \pm 0.65$ W m⁻² K⁻¹ (5-95%). The CICERO-SCM `lambda` parameter is the inverse: a 5-95% range on feedback `[0.51, 1.81]` W m⁻² K⁻¹ inverts to `lambda ∈ [0.55, 1.96]` K (W m⁻²)⁻¹. We pad slightly to bracket the 99% tail and to leave the cscm-calibrate prior the freedom to shape the posterior.

In [3]:
ar6_feedback_central = 1.16
ar6_feedback_5_95_halfwidth = 0.65   # 5-95%, so 1.645 sigma
feedback_lo_5pct = ar6_feedback_central - ar6_feedback_5_95_halfwidth   # 0.51
feedback_hi_95pct = ar6_feedback_central + ar6_feedback_5_95_halfwidth  # 1.81

lambda_hi = round(1.0 / feedback_lo_5pct, 2)   # large lambda = weak feedback (high sensitivity)
lambda_lo = round(1.0 / feedback_hi_95pct, 2)

# Pad ~10-20% for prior breadth.
lambda_prior = [round(lambda_lo * 0.9, 2), round(lambda_hi * 1.1, 2)]

print(f'IPCC AR6 long-term feedback 5-95%: {feedback_lo_5pct:.2f} to {feedback_hi_95pct:.2f} W m^-2 K^-1')
print(f'Inverted to lambda (K per W m^-2):  {lambda_lo:.2f} to {lambda_hi:.2f}')
print(f'With ~10-20% padding: lambda prior  {lambda_prior}')
print()
print(f'Compare with current cscm-calibrate test config: lambda = [0.4, 1.3]')
print(f'  -> current upper bound 1.3 corresponds to feedback {1/1.3:.2f} W m^-2 K^-1,')
print(f'     which truncates the IPCC AR6 5-95% lower-feedback / high-sensitivity tail.')

IPCC AR6 long-term feedback 5-95%: 0.51 to 1.81 W m^-2 K^-1
Inverted to lambda (K per W m^-2):  0.55 to 1.96
With ~10-20% padding: lambda prior  [0.5, 2.16]

Compare with current cscm-calibrate test config: lambda = [0.4, 1.3]
  -> current upper bound 1.3 corresponds to feedback 0.77 W m^-2 K^-1,
     which truncates the IPCC AR6 5-95% lower-feedback / high-sensitivity tail.


## Prior on `delta_lambda_aero`

Andrews' pattern effect is  the difference between the satellite-era effective feedback and the long-term equilibrium feedback. From Andrews v5 windows vs the IPCC AR6 long-term central value, $\Delta\lambda$ spans roughly $0.28$ to $0.82$ W m⁻² K⁻¹. Forster et al. (AR6) and Sherwood et al. (2020) give an assessed range of $0.5 \pm 0.5$ W m⁻² K⁻¹, which extends to essentially zero on the lower end (consistent with Lewis & Mauritsen's HadISST-based 'no pattern effect' result).

The CICERO-SCM parameter relates to this as

$$\Delta\lambda_\text{aero} = \frac{\Delta\lambda}{\langle w_\text{aero}\rangle_\text{satellite era}}$$

so we need the model-empirical $\langle w_\text{aero}\rangle$ over the Andrews windows. We extract this from one CSCM run pair, exploiting the fact that `delta_lambda_aero` enters $\lambda_\text{eff}$ linearly: two runs at $\Delta\lambda_\text{aero} = 0$ and $1$ differ in their diagnosed $-\lambda_\text{hist}$ over a window by exactly $\langle w_\text{aero}\rangle$ over that window.

In [4]:
gaspam = read_components(os.path.join(TEST_DATA, 'gases_v1RCMIP.txt'))
df_nat_ch4 = read_natural_emissions(os.path.join(TEST_DATA, 'natemis_ch4.txt'), 'CH4')
df_nat_n2o = read_natural_emissions(os.path.join(TEST_DATA, 'natemis_n2o.txt'), 'N2O')
df_ssp2_conc = read_inputfile(os.path.join(TEST_DATA, 'ssp245_conc_RCMIP.txt'))
emi_input = read_inputfile(os.path.join(TEST_DATA, 'ssp245_em_RCMIP.txt'))
emi_input.rename(columns={'CO2': 'CO2_FF', 'CO2.1': 'CO2_AFOLU'}, inplace=True)

NYSTART, NYEND = 1750, 2100
years = np.arange(NYSTART, NYEND + 1)

PAMSET_UDM = dict(
    threstemp=7.0, rlamdo=16.0, akapa=0.634, cpi=0.4,
    W=4, beto=3.5, mixed=60.0, foan=0.61, foas=0.81,
    ebbeta=0.0, fnso=0.7531, lm=40, ldtime=12, **{'lambda': 0.86},
)
PAMSET_EMICONC = {
    'qbmb': 0.0, 'qo3': 0.5, 'qdirso2': -0.00308,
    'qindso2': -0.97 / 57.052577209999995,
    'qbc': 0.0279, 'qoc': -0.00433, 'qh2o_ch4': 0.091915, 'ref_yr': 2010,
}

def cscm_run(delta_lambda_aero):
    cscm = CICEROSCM({
        'gaspam_data': gaspam, 'emstart': 1751, 'conc_run': False,
        'nystart': NYSTART, 'nyend': NYEND,
        'concentrations_data': df_ssp2_conc, 'emissions_data': emi_input,
        'nat_ch4_data': df_nat_ch4, 'nat_n2o_data': df_nat_n2o, 'idtm': 24,
    })
    pamset = dict(PAMSET_UDM)
    pamset['delta_lambda_aero'] = delta_lambda_aero
    cscm._run({'results_as_dict': True}, pamset_udm=pamset, pamset_emiconc=PAMSET_EMICONC)
    return pd.DataFrame({
        'T': np.asarray(cscm.results['dT_glob']).squeeze(),
        'N': np.asarray(cscm.results['RIB_glob']).squeeze(),
        'F': np.asarray(cscm.results['Total_forcing']).squeeze(),
    }, index=pd.Index(years, name='year'))

In [5]:
def neg_lambda_hist(df, y0, y1, exclude=PINATUBO_YEARS):
    """OLS slope of (F - N) on T over [y0, y1], excluding given years (Andrews method)."""
    sub = df.loc[y0:y1].drop(index=list(exclude), errors='ignore')
    T = sub['T'].values
    R = sub['N'].values - sub['F'].values
    Tc, Rc = T - T.mean(), R - R.mean()
    return -float((Tc * Rc).sum() / (Tc * Tc).sum())

df_d0 = cscm_run(0.0)
df_d1 = cscm_run(1.0)

rows = []
for _, r in v5.iterrows():
    nl0 = neg_lambda_hist(df_d0, int(r['y0']), int(r['y1']))
    nl1 = neg_lambda_hist(df_d1, int(r['y0']), int(r['y1']))
    rows.append({'window': r['window'], 'w_aero_mean': nl1 - nl0})

w_aero_table = pd.DataFrame(rows)
print('Empirical <w_aero> over Andrews v5 windows (CICERO-SCM, SSP2-4.5):')
print(w_aero_table.to_string(index=False, float_format='%.3f'))
w_aero_central = float(w_aero_table['w_aero_mean'].mean())
w_aero_lo = float(w_aero_table['w_aero_mean'].min())
w_aero_hi = float(w_aero_table['w_aero_mean'].max())
print(f'\n<w_aero> central (window mean):  {w_aero_central:.3f}')
print(f'<w_aero> range across v5 windows: [{w_aero_lo:.3f}, {w_aero_hi:.3f}]')

emstart can not be changed for same instance of ConcentrationsEmisssionsHandler. Resetting with old value 1751. If you want to run with a different value, please create a separate instance


emstart can not be changed for same instance of ConcentrationsEmisssionsHandler. Resetting with old value 1751. If you want to run with a different value, please create a separate instance


Empirical <w_aero> over Andrews v5 windows (CICERO-SCM, SSP2-4.5):
   window  w_aero_mean
1985-2014        0.380
1986-2015        0.371
1987-2016        0.359
1988-2017        0.345
1989-2018        0.328
1990-2019        0.310

<w_aero> central (window mean):  0.349
<w_aero> range across v5 windows: [0.310, 0.380]


In [6]:
# Translate the literature range on the pattern effect into bounds on
# delta_lambda_aero, the model parameter.
#
# The "pattern effect" Delta_lambda is the offset between the effective
# climate feedback diagnosed over the satellite-era historical record and
# the long-term feedback associated with equilibrium climate sensitivity.
# It exists because the observed warming pattern (warm-pool warming,
# eastern-Pacific and Southern-Ocean cooling) is not the pattern that
# emerges in long-term CO2-forced equilibration, so historical feedbacks
# are not directly informative about ECS without an adjustment.
#
# Two literature anchors:
#
#  - Forster et al. (2021, IPCC AR6 WG1 Ch.7) and Sherwood et al. (2020,
#    Rev. Geophys., 58, e2019RG000678) assess Delta_lambda = 0.5 +/- 0.5
#    W m^-2 K^-1 (5-95%). The lower end straddles zero, consistent with
#    Lewis & Mauritsen (2021, J. Climate, 34, 39-55) who report a
#    "negligible unforced historical pattern effect" using HadISST1
#    boundary conditions.
#
#  - Andrews et al. (2022, JGR Atmospheres, 127, e2022JD036675) Table 4:
#    DEEP-C v5 windows give -lambda_hist in [1.44, 1.98]; subtracting the
#    IPCC AR6 long-term -lambda_4xCO2 = 1.16 gives Delta_lambda in
#    [0.28, 0.82] across windows. Tighter than the Forster/Sherwood
#    assessment because it uses a single forward model over fixed SSTs.
#
# We bracket the more conservative Forster/Sherwood range, allowing
# slightly negative values for the Lewis & Mauritsen tail.

delta_lit_lo = -0.2   # below zero to allow Lewis-Mauritsen 'no pattern effect'
delta_lit_hi = 1.0    # Forster/Sherwood AR6 5-95% upper

# Translate via Delta_lambda_aero = Delta_lambda / <w_aero>.
delta_aero_lo = delta_lit_lo / w_aero_central
delta_aero_hi = delta_lit_hi / w_aero_central

# Pad ~20% for prior breadth (and round to one decimal).
delta_aero_prior = [round(delta_aero_lo * 1.2, 1), round(delta_aero_hi * 1.2, 1)]

print(f'Pattern-effect literature range (W m^-2 K^-1):  [{delta_lit_lo:+.2f}, {delta_lit_hi:+.2f}]')
print(f'Divide by <w_aero> = {w_aero_central:.3f}:')
print(f'  -> delta_lambda_aero            [{delta_aero_lo:+.2f}, {delta_aero_hi:+.2f}]')
print(f'  -> with ~20% padding for prior  {delta_aero_prior}')

Pattern-effect literature range (W m^-2 K^-1):  [-0.20, +1.00]
Divide by <w_aero> = 0.349:
  -> delta_lambda_aero            [-0.57, +2.87]
  -> with ~20% padding for prior  [-0.7, 3.4]


## Summary — drop-in `prior_distro_dict` snippet

Add or update the following entries in `prior_configs.prior_distro_dict` of the cscm-calibrate config. The bounds bracket the physically motivated literature ranges with mild padding, leaving the calibration's existing constraints (GMST, OHC, ERFaer, atmospheric CO₂, ocean and biosphere carbon flux) to shape the posterior.

In [7]:
snippet = {
    'lambda': lambda_prior,
    'delta_lambda_aero': delta_aero_prior,
}
print(json.dumps(snippet, indent=4))
print()
print('Justification, in one line each:')
print(f'  lambda           : IPCC AR6 long-term feedback 5-95% [0.51, 1.81] inverted, with ~10-20% padding - effective ECS range 1.85 to 8K.')
print(f'  delta_lambda_aero: Forster/Sherwood AR6 pattern effect [-0.2, 1.0] W m^-2 K^-1 divided by')
print(f'                     model-empirical <w_aero> = {w_aero_central:.3f} over Andrews v5 satellite-era windows,')
print(f'                     with ~20% padding.')

{
    "lambda": [
        0.5,
        2.16
    ],
    "delta_lambda_aero": [
        -0.7,
        3.4
    ]
}

Justification, in one line each:
  lambda           : IPCC AR6 long-term feedback 5-95% [0.51, 1.81] inverted, with ~10-20% padding - effective ECS range 1.85 to 8K.
  delta_lambda_aero: Forster/Sherwood AR6 pattern effect [-0.2, 1.0] W m^-2 K^-1 divided by
                     model-empirical <w_aero> = 0.349 over Andrews v5 satellite-era windows,
                     with ~20% padding.
